In [9]:
using DataFrames
using JSON
using Turing

In [2]:
DATA = "../../data"

"../../data"

In [3]:
englandleague = JSON.parsefile("$DATA/matches_England.json")
matchesdf = DataFrame(home=[], away=[], scorehome=[], scoreaway=[])

,home,away,scorehome,scoreaway
,Any,Any,Any,Any


In [4]:
englandleague[1]

Dict{String,Any} with 14 entries:
  "label"         => "Burnley - AFC Bournemouth, 1 - 2"
  "teamsData"     => Dict{String,Any}("1646"=>Dict{String,Any}("coachId"=>8880,…
  "wyId"          => 2500089
  "duration"      => "Regular"
  "status"        => "Played"
  "venue"         => "Turf Moor"
  "roundId"       => 4405654
  "winner"        => 1659
  "gameweek"      => 38
  "date"          => "May 13, 2018 at 4:00:00 PM GMT+2"
  "competitionId" => 364
  "dateutc"       => "2018-05-13 14:00:00"
  "seasonId"      => 181150
  "referees"      => Any[Dict{String,Any}("role"=>"referee","refereeId"=>385705…

In [5]:
matches = []
for match in englandleague
    push!(matches, split(match["label"], ","))
end

In [6]:
for match in matches
    home, away = split(match[1], " - ")
    scorehome, scoreaway = split(match[2], " - ")
    push!(matchesdf, 
          [home, away, parse(Int, scorehome), parse(Int, scoreaway)])
end

In [7]:
first(matchesdf, 5)

,home,away,scorehome,scoreaway
,Any,Any,Any,Any
1,Burnley,AFC Bournemouth,1,2
2,Crystal Palace,West Bromwich Albion,2,0
3,Huddersfield Town,Arsenal,0,1
4,Liverpool,Brighton & Hove Albion,4,0
5,Manchester United,Watford,1,0


In [8]:
teams = unique(collect(matchesdf[:, :home]))

20-element Array{Any,1}:
 "Burnley"
 "Crystal Palace"
 "Huddersfield Town"
 "Liverpool"
 "Manchester United"
 "Newcastle United"
 "Southampton"
 "Swansea City"
 "Tottenham Hotspur"
 "West Ham United"
 "Manchester City"
 "Leicester City"
 "Chelsea"
 "Arsenal"
 "Everton"
 "AFC Bournemouth"
 "Watford"
 "West Bromwich Albion"
 "Stoke City"
 "Brighton & Hove Albion"

### Create the Model

In [18]:
@model function games(hometeams, awayteams, scorehome, scoreaway, teams)
    # hyper priors
    sigatt ~ Exponential(1)
    sigdef ~ Exponential(1)
    muatt ~ Normal(0, 0.1)
    mudef ~ Normal(0, 0.1)
    home ~ Normal(0, 1)
    
    # Team-specific effects
    att ~ filldist(Normal(muatt, sigatt), length(teams))
    def ~ filldist(Normal(mudef, sigdef), length(teams))
    dict = Dict{String, Int64}()
    for (i, team) in enumerate(teams)
        dict[team] = i
    end
    
    # Zero-sum constraints
    offset = mean(att) + mean(def)
    logthetahome = Vector{Real}(undef, length(hometeams))
    logthetaaway = Vector{Real}(undef, length(awayteams))
    
    # Modeling score-rate and scores
    for i in 1:length(hometeams)
        # score rates
        logthetahome[i] = (home 
                           + att[dict[hometeams[i]]] 
                           + def[dict[awayteams[i]]]
                           - offset)
        logthetaaway[i] = (
            att[dict[awayteams[i]]] + def[dict[hometeams[i]]] - offset)
        # scores
        scorehome[i] ~ LogPoisson(logthetahome[i])
        scoreaway[i] ~ LogPoisson(logthetaaway[i])
    end
end

games (generic function with 1 method)

In [19]:
model = games(matchesdf[:, :home], 
              matchesdf[:, :away],
              matchesdf[:, :scorehome], 
              matchesdf[:, :scoreaway], 
              teams)

DynamicPPL.Model{var"#7#8",(:hometeams, :awayteams, :scorehome, :scoreaway, :teams),(),(),NTuple{5,Array{Any,1}},Tuple{}}(:games, var"#7#8"(), (hometeams = Any["Burnley", "Crystal Palace", "Huddersfield Town", "Liverpool", "Manchester United", "Newcastle United", "Southampton", "Swansea City", "Tottenham Hotspur", "West Ham United"  …  "Manchester United", "Newcastle United", "Brighton & Hove Albion", "Chelsea", "Crystal Palace", "Everton", "Southampton", "West Bromwich Albion", "Watford", "Arsenal"], awayteams = Any["AFC Bournemouth", "West Bromwich Albion", "Arsenal", "Brighton & Hove Albion", "Watford", "Chelsea", "Manchester City", "Stoke City", "Leicester City", "Everton"  …  "West Ham United", "Tottenham Hotspur", "Manchester City", "Burnley", "Huddersfield Town", "Stoke City", "Swansea City", "AFC Bournemouth", "Liverpool", "Leicester City"], scorehome = Any[1, 2, 0, 4, 1, 3, 0, 1, 5, 3  …  4, 0, 0, 2, 0, 1, 0, 1, 3, 4], scoreaway = Any[2, 0, 1, 0, 0, 0, 1, 2, 4, 1  …  0, 2, 2, 

In [20]:
posterior = sample(model, NUTS(), 3000)

┌ Info: Found initial step size
│   ϵ = 0.2
└ @ Turing.Inference /Users/dsatterthwaite/.julia/packages/Turing/PyTy2/src/inference/hmc.jl:188
Sampling: 100%|█████████████████████████████████████████| Time: 0:01:53


LoadError: InterruptException:

### Analyzing the Results